# K10 · Final evaluation: ablation + speed + answers (final system)
Three parts, each saved to Hugging Face as soon as it finishes:
1. **Retrieval ablation** (no Gemini): switch off one retrieval step at a time on the same 100 QASPER + 50 SPIQA questions.
2. **Speed**: time to process the two demo papers on the GPU and in website-like CPU mode (2 threads), time to open saved
   results (demo paper), retrieval and Gemini answer time.
3. **Answer evaluation with the final system** (paragraph-only re-ranking): QASPER answer F1 / evidence F1 / citation
   precision and SPIQA judged correctness, same questions as K8.

Needs: **GPU T4 x2**, **Internet on**, secrets **`HF_TOKEN`** (write) and **`GEMINI_API_KEY`**.
Optional secret **`GEMINI_API_KEY_2`**: a key from a *second* Google Cloud project (its own free daily quota); used only
when the first key's quota is used up. About 2–2.5 hours. If it stops, run all cells again: every part continues
where it stopped.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. App packages, Gemini keys, upload helper

In [ ]:
!pip install -q -r requirements-app.txt
from kaggle_secrets import UserSecretsClient
for name in ("GEMINI_API_KEY", "GEMINI_API_KEY_2"):
    try:
        os.environ[name] = UserSecretsClient().get_secret(name); print(name, "loaded")
    except Exception:
        print(name, "not attached" + (" (optional)" if name.endswith("_2") else " -> part 3 cannot run"))
from huggingface_hub import HfApi
from IPython.display import Markdown, display
def upload(folder, path_in_repo, msg):
    HfApi().upload_folder(repo_id="Kavinvelavan/researchlens-results", repo_type="dataset", folder_path=folder,
                          path_in_repo=path_in_repo, token=os.environ.get("HF_TOKEN"), commit_message=msg,
                          ignore_patterns=["qdrant*/*", "papers/*", "work_*/*", "*.pdf"])
    print("uploaded", path_in_repo)

## 2. Retrieval ablation (no Gemini, about 25 minutes)

In [ ]:
AB = "/kaggle/working/ablation"
!python -m researchlens.evaluation.ablation --out_dir {AB} --data_dir /kaggle/working/eval_data --n_qasper 100 --n_spiqa 50
display(Markdown(open(f"{AB}/report.md").read()))
upload(AB, "results/ablation", "retrieval ablation")

## 3. Speed (about 20–30 minutes)
The two demo papers from arXiv. First on the GPU (with 3 Gemini answers each), then in website-like CPU mode:
no GPU, 2 PyTorch threads, the website's limits (`RL_LOW_MEMORY=1`).

In [ ]:
TM = "/kaggle/working/timing"
os.makedirs(TM, exist_ok=True)
!wget -q -O {TM}/swin.pdf https://arxiv.org/pdf/2103.14030v2
!wget -q -O {TM}/vit.pdf https://arxiv.org/pdf/2010.11929v2
!ls -la {TM}
!python -m researchlens.evaluation.timing {TM}/swin.pdf {TM}/vit.pdf --label gpu --out {TM} --answers 3
!CUDA_VISIBLE_DEVICES= RL_LOW_MEMORY=1 python -m researchlens.evaluation.timing {TM}/swin.pdf {TM}/vit.pdf --label cpu_2threads --out {TM}
upload(TM, "results/timing", "speed measurements")

## 4. Answer evaluation with the final system (Gemini, about 1–1.5 hours)

In [ ]:
FIN = "/kaggle/working/system_eval_final"
!python -m researchlens.evaluation.system_eval --out_dir {FIN} --data_dir /kaggle/working/eval_data --n_qasper 100 --n_spiqa 50
display(Markdown(open(f"{FIN}/report.md").read()))
upload(FIN, "results/system_eval_final", "final system evaluation (paragraph-only re-ranking)")